# Multi Agent Fork join logic

In [ ]:

from langgraph.graph import StateGraph, END
from langchain_core.runnables import RunnableLambda
from langchain_cohere import ChatCohere
from typing import TypedDict
import os

# === Setup Cohere ===
api_key_prod = "uUlulV3HkN4ti01lrNIS6rwYgHoPkKInUoWVLBjr"
llm = ChatCohere(model="command-r", temperature=0.4, cohere_api_key=api_key_prod)

# === Define Shared State ===
class FinanceState(TypedDict, total=False):
    step: str
    goal: str
    risk_profile: str
    income: str
    timeline_estimate: str
    tax_advice: str
    suggestion: str

# === Agent 1: Goal Clarifier ===
def goal_clarifier(state):
    res = llm.invoke("You are GoalClarifierAgent. Ask the user their primary financial goal.Present him with 2-3 example goals.Keep naration to few words.")
    print(f"\n🎯 GoalClarifierAgent: {res.content}")
    goal = input("User: ")
    return {"goal": goal, "step": "risk_assessor"}

# === Agent 2: Risk Assessor ===
def risk_assessor(state):
    res = llm.invoke("You are RiskAssessorAgent. Ask the user to describe their risk appetite: low, moderate, or high.Keep naration to few words.")
    print(f"\n📊 RiskAssessorAgent: {res.content}")
    risk = input("User: ")
    return {"risk_profile": risk, "step": "budget_analyzer"}

# === Agent 3: Budget Analyzer ===
def budget_analyzer(state):
    res = llm.invoke("You are BudgetAnalyzerAgent. Ask the user for their monthly income or savings available for investment.Keep naration to few words.")
    print(f"\n💵 BudgetAnalyzerAgent: {res.content}")
    income = input("User: ")
    return {"income": income, "step": "fork_to_parallel"}

# === Agent 4a: Timeline Estimator ===
def timeline_estimator(state):
    prompt = (
        f"You are TimelineEstimatorAgent. Estimate how long it would take to achieve the financial goal '{state['goal']}' "
        f"with a monthly income of '{state['income']}' and a '{state['risk_profile']}' risk profile. "
        f"Give the estimate in years and describe the rationale."
        f"Keep the answer crisp in 2-3 lines"
    )
    res = llm.invoke(prompt)
    print(f"\n📅 TimelineEstimatorAgent: {res.content}")
    return {"timeline_estimate": res.content}

# === Agent 4b: Tax Planner ===
def tax_planner(state):
    prompt = (
        f"You are TaxPlannerAgent. Provide New York state-specific tax advice for the goal '{state['goal']}' "
        f"given a monthly income of '{state['income']}' and risk profile '{state['risk_profile']}'."
        f"Keep the answer crisp in 2-3 lines"
    )
    res = llm.invoke(prompt)
    print(f"\n🧾 TaxPlannerAgent: {res.content}")
    return {"tax_advice": res.content}

# === Agent 5: Plan Recommender (after parallel join) ===
def plan_recommender(state):
    prompt = (
        f"You are PlanRecommenderAgent. Based on the goal '{state['goal']}', risk '{state['risk_profile']}', "
        f"income '{state['income']}', timeline estimate '{state['timeline_estimate']}', and tax advice '{state['tax_advice']}', "
        f"provide a 2–3 sentence personalized investment recommendation."
    )
    res = llm.invoke(prompt)
    print(f"\n💡 PlanRecommenderAgent: {res.content}")
    return {"suggestion": res.content}

# === Building LangGraph with Fork/Join ===
graph = StateGraph(FinanceState)

# Add nodes
graph.add_node("goal_clarifier", RunnableLambda(goal_clarifier))
graph.add_node("risk_assessor", RunnableLambda(risk_assessor))
graph.add_node("budget_analyzer", RunnableLambda(budget_analyzer))
graph.add_node("timeline_estimator", RunnableLambda(timeline_estimator))
graph.add_node("tax_planner", RunnableLambda(tax_planner))
graph.add_node("plan_recommender", RunnableLambda(plan_recommender))

# Entry + sequential steps
graph.set_entry_point("goal_clarifier")
graph.add_edge("goal_clarifier", "risk_assessor")
graph.add_edge("risk_assessor", "budget_analyzer")

# === Fork into parallel branches ===
# After budget_analyzer, run both timeline_estimator and tax_planner
graph.add_edge("budget_analyzer", "timeline_estimator")
graph.add_edge("budget_analyzer", "tax_planner")

# === Join after both timeline and tax advice are done ===
# Both write to shared state and then go to plan_recommender
graph.add_edge("timeline_estimator", "plan_recommender")
graph.add_edge("tax_planner", "plan_recommender")

# Final node
graph.add_edge("plan_recommender", END)

# Compile graph
multi_agent = graph.compile()

# === Visualize Graph (Optional) ===
from IPython.display import Image, display
display(Image(multi_agent.get_graph().draw_mermaid_png()))

# === Run the Multi-Agent System ===
if __name__ == "__main__":
   
    multi_agent.invoke({})
